In [ ]:
from neo4j import GraphDatabase
import json

# ⚠️ UPDATE THESE with your Neo4j credentials
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "LegalPassword123"

# --- Connect ---
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"✅ Connected to Neo4j at {NEO4J_URI}")

# --- Load triples ---
with open("extracted_triples (2).json", "r") as f:
    triples = json.load(f)
print(f"📂 Loaded {len(triples)} triples from file")

# --- Clear existing graph (fresh start) ---
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("🗑️  Cleared existing graph")

# --- Ingest triples one by one ---
loaded = 0
skipped = 0

with driver.session() as session:
    for t in triples:
        source_id = t.get("source_id", "UNKNOWN")
        target = t.get("target_citation")
        action = t.get("action", "RELATES_TO")

        # Skip if target is null
        if not target:
            skipped += 1
            continue

        # Determine source type from the ID
        if any(prefix in source_id for prefix in ["uksc_", "ewca_", "ewhc_", "ukut_"]):
            source_type = "CaseLaw"
        else:
            source_type = "Legislation"

        # Cypher: create source node, target node, and relationship
        query = """
        MERGE (s:LegalDoc {id: $source_id})
        SET s.type = $source_type
        MERGE (t:LegalDoc {citation: $target})
        MERGE (s)-[r:LEGAL_RELATIONSHIP {action: $action}]->(t)
        SET r.detail = $detail,
            r.date = $date,
            r.action_type = $action
        """
        session.run(query,
            source_id=source_id,
            source_type=source_type,
            target=target,
            action=action,
            detail=t.get("detail_text"),
            date=t.get("effective_date")
        )
        loaded += 1

        if loaded % 50 == 0:
            print(f"   ... loaded {loaded} triples")

print(f"\n{'='*50}")
print(f"✅ NEO4J INGESTION COMPLETE")
print(f"   Loaded: {loaded} triples")
print(f"   Skipped (null targets): {skipped}")

# --- Verify: print graph stats ---
with driver.session() as session:
    nodes = session.run("MATCH (n:LegalDoc) RETURN count(n) as cnt").single()["cnt"]
    edges = session.run("MATCH ()-[r]->() RETURN count(r) as cnt").single()["cnt"]
    actions = session.run("""
        MATCH ()-[r:LEGAL_RELATIONSHIP]->()
        RETURN r.action_type AS action, count(*) AS count
        ORDER BY count DESC
    """)
    action_counts = {r["action"]: r["count"] for r in actions}

print(f"\n📊 Graph Stats:")
print(f"   Nodes: {nodes}")
print(f"   Edges: {edges}")
print(f"   Actions: {action_counts}")

driver.close()
print("\n✅ Neo4j connection closed")

In [1]:
print("Hello World")

Hello World
